In [55]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [56]:
files = [
    "CRMLSSold202511.csv",
    "CRMLSSold202512.csv",
    "CRMLSSold202601.csv",
    "CRMLSSold202602.csv",
    "CRMLSSold202603.csv",
    "CRMLSSold202604.csv",
    "CRMLSSold202605.csv"
]

dfs = [pd.read_csv(file) for file in files]

df = pd.concat(dfs, ignore_index=True)

/tmp/ipykernel_1055/4166851852.py:11: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs = [pd.read_csv(file) for file in files]


In [57]:
df = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"] == "SingleFamilyResidence")
]

### Inspect Missing Values

In [58]:
print(len(df))
print("\nMissing values after preprocessing:")
print(df.isnull().sum()[df.isnull().sum() > 0])

71466

Missing values after preprocessing:
BuyerAgentAOR                    3665
ListAgentAOR                       19
Flooring                        25974
ViewYN                           6211
WaterfrontYN                    71429
                                ...  
HighSchoolDistrict              19210
PostalCode                          1
AssociationFee                  20451
LotSizeSquareFeet                1227
MiddleOrJuniorSchoolDistrict    71466
Length: 61, dtype: int64


In [59]:
# check how many missing values each boolean column has
cols = [
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "NewConstructionYN"
]

print(df[cols].isna().sum())

ViewYN               6211
PoolPrivateYN        5585
AttachedGarageYN     8687
FireplaceYN            63
NewConstructionYN    5396
dtype: int64



### Handling missing values








In [60]:
# Drop columns where more than 70% of values are missing
threshold = 0.7
df = df.loc[:, df.isnull().mean() < threshold]

In [61]:
# drop rows that have missing values in the ClosePrice column
df = df.dropna(subset=["ClosePrice"])

# Remove observations where ClosePrice is less than or equal to 0
df = df[df['ClosePrice'] > 0]

In [62]:
# Remove duplicate rows
df.drop_duplicates(inplace=True)

In [63]:
# flag and impute the AttachedGarageYN column (the only boolean column with missing values)
df["AttachedGarageYN_missing"] = df["AttachedGarageYN"].isna().astype(int)

df["AttachedGarageYN"] = df["AttachedGarageYN"].fillna(df["AttachedGarageYN"].mode()[0])

/tmp/ipykernel_1055/3989187805.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["AttachedGarageYN"] = df["AttachedGarageYN"].fillna(df["AttachedGarageYN"].mode()[0])


In [64]:
# force key numeric columns into numbers
# then, fill missing values with the Median
core_features = ["LivingArea", "LotSizeArea", "BedroomsTotal", "BathroomsTotalInteger"]
for col in core_features:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(df[col].median())


In [65]:
# replace missing values in the City and CountyOrParish columns with "Unknown"
df["City"] = df["City"].fillna("Unknown")
df["CountyOrParish"] = df["CountyOrParish"].fillna("Unknown")

In [66]:
# replace missing values of the YearBuilt column with median
# then, create a new column for age of the properties

df["YearBuilt_missing"] = df["YearBuilt"].isna().astype(int)

df["YearBuilt"] = df["YearBuilt"].fillna(df["YearBuilt"].median())

df["Age"] = 2026 - df["YearBuilt"]

### Convert categorical fields to numeric (encoding)

In [67]:
df = pd.get_dummies(df, columns=["City"], drop_first=True)

In [68]:

df = pd.get_dummies(df, columns=["CountyOrParish"], drop_first=True)


### cleaned CSV:

In [74]:
df.to_csv("cleaned_dataset.csv", index=False)

### Create train/test split

In [70]:
df["CloseDate"] = pd.to_datetime(df["CloseDate"])
df["Month"] = df["CloseDate"].dt.to_period("M")

months = sorted(df["Month"].unique())

X = 3

test_month = months[-1]
train_months = months[-(X + 1):-1]

train_df = df[df["Month"].isin(train_months)].copy()
test_df = df[df["Month"] == test_month].copy()

In [71]:
num_cols = ["LivingArea", "LotSizeArea", "BedroomsTotal", "BathroomsTotalInteger"]

In [72]:
# fit scaler on train data only
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

train_df[num_cols] = scaler.fit_transform(train_df[num_cols])

In [73]:
test_df[num_cols] = scaler.transform(test_df[num_cols])